# 02 Preprocessing
Tokenization of the texts, the choice of the word embeddings (GloVe), lowercasing and stopwords, and the input of both models: for model 1 the average GloVe vector of each text, saved to `features/model1.npz`; for model 2 the padded word numbers of each text and the GloVe embedding matrix, saved to `features/model2.npz`.

Load the train, validation and test splits (like on slide 9 of the course project slides).

In [1]:
from datasets import load_dataset

train = load_dataset("amarfurt/art-nlp-hs26", split="train")
valid = load_dataset("amarfurt/art-nlp-hs26", split="validation")
test = load_dataset("amarfurt/art-nlp-hs26", split="test")

print(len(train), len(valid), len(test))

16801 1000 1532


## Tokenization
NLTK's tokenization needs the `punkt_tab` resource, a pretrained model that `word_tokenize` uses to split text into sentences. It is downloaded first.


In [2]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mmici\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Compare splitting at whitespace with NLTK's `word_tokenize` on one example (the first short training hypothesis that contains both a comma and a contraction).


In [3]:
text = train[269]['hypothesis_1']
print(text)
print(text.split())
print(nltk.word_tokenize(text))

He didn't listen to them, and had to go to the hospital.
['He', "didn't", 'listen', 'to', 'them,', 'and', 'had', 'to', 'go', 'to', 'the', 'hospital.']
['He', 'did', "n't", 'listen', 'to', 'them', ',', 'and', 'had', 'to', 'go', 'to', 'the', 'hospital', '.']


`split()` cuts the text only at spaces, so punctuation stays attached to words (`them,`, `hospital.`). `word_tokenize` separates punctuation (`them` and `,`) and contractions (`did` and `n't`) from the words. This way the words match the vocabulary of the word embeddings, and negations like `n't` stay their own tokens. `word_tokenize` is used for all texts.

## Word embeddings
Load the pretrained GloVe embeddings (300 dimensions) with gensim's downloader, and look at the vector size, the vocabulary size and the first words.

In [4]:
import gensim.downloader as api
wv = api.load('glove-wiki-gigaword-300')


In [5]:
print(f'Vector size: {wv.vector_size}')
print(f'Vocabulary size: {len(wv.index_to_key)}')
for index, word in enumerate(wv.index_to_key[:10]):
    print(f"word #{index}: {word}")

Vector size: 300
Vocabulary size: 400000
word #0: the
word #1: ,
word #2: .
word #3: of
word #4: to
word #5: and
word #6: in
word #7: a
word #8: "
word #9: 's


Count the words in the training data. Tokenize all texts (O1, O2, h1, h2) and count how many words there are in total (tokens) and how many distinct words (types), once as written and once lowercased. This gives the list of words that need a vector.

In [6]:
columns = ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2']

tokens = []
for column in columns:
    for text in train[column]:
        tokens.extend(nltk.word_tokenize(text))

fdist = nltk.FreqDist(tokens)
fdist_lower = nltk.FreqDist(token.lower() for token in tokens)
print('Tokens:', len(tokens))
print('Types:', len(fdist))
print('Types (lowercased):', len(fdist_lower))
print(f'Type-token-ratio: {len(fdist) / len(tokens):.2%}')

Tokens: 655477
Types: 17789
Types (lowercased): 16210
Type-token-ratio: 2.71%


Share of types and tokens that have a vector in GloVe, and the most frequent words without a vector.


In [7]:
def coverage(wv, fdist):
    known = [word for word in fdist if word in wv]
    known_tokens = sum(fdist[word] for word in known)
    print(f'Known types: {len(known) / len(fdist):.2%}')
    print(f'Known tokens: {known_tokens / fdist.N():.2%}')
    unknown = [word for word, count in fdist.most_common() if word not in wv]
    print('Most frequent unknown:', unknown[:15])

print('GloVe, cased:')
coverage(wv, fdist)
print('GloVe, lowercased:')
coverage(wv, fdist_lower)


GloVe, cased:
Known types: 73.82%
Known tokens: 87.43%
Most frequent unknown: ['I', 'The', 'He', 'She', 'My', 'It', 'A', 'They', 'Gina', 'Amy', 'Bob', 'Tim', 'Sam', 'We', 'John']
GloVe, lowercased:
Known types: 95.40%
Known tokens: 99.84%
Most frequent unknown: ['mturk', '..', 'selfie', 'zaiden', 'a+', 'antonela', 'fariah', 'myah', 'shalil', 'rubix', 'kelsea', 'jacelyn', 'lamay', 'marisse', 'lakyn']


GloVe only contains lowercase words: cased, only 87.43% of the tokens have a vector; lowercased, 99.84%, so almost every word the model sees has a vector. The remaining unknown words are rare names, typos and *mturk*. **All texts are therefore lowercased.**

For comparison: the coverage of fastText and word2vec (the embeddings from the course exercise). Each is loaded, measured and deleted again to free memory.

In [8]:
for name in ['fasttext-wiki-news-subwords-300', 'word2vec-google-news-300']:
    other_wv = api.load(name)
    print(f'{name}, cased:')
    coverage(other_wv, fdist)
    print(f'{name}, lowercased:')
    coverage(other_wv, fdist_lower)
    del other_wv


fasttext-wiki-news-subwords-300, cased:
Known types: 96.95%
Known tokens: 99.38%
Most frequent unknown: ["n't", '``', "''", 'mTurk', 'Zaiden', 'mturk', 'A+', 'Antonela', 'Fariah', 'Mturk', 'Shalil', 'rubix', 'Marisse', 'Lakyn', 'Tanice']
fasttext-wiki-news-subwords-300, lowercased:
Known types: 93.97%
Known tokens: 99.11%
Most frequent unknown: ["n't", '``', "''", 'kelsi', 'marcy', 'rhonda', 'joanie', 'mindy', 'mturk', 'horace', 'tabitha', 'hallie', 'horatio', 'juanita', 'callie']
word2vec-google-news-300, cased:
Known types: 94.12%
Known tokens: 78.34%
Most frequent unknown: ['.', 'to', 'a', 'and', ',', 'of', "'s", '!', '``', "'", '10', '20', 'didnt', '100', 'cancelled']
word2vec-google-news-300, lowercased:
Known types: 88.86%
Known tokens: 77.62%
Most frequent unknown: ['.', 'to', 'a', 'and', ',', 'of', "'s", '!', '``', "'", '10', '20', 'didnt', 'tami', 'maddie']


Conclusion: GloVe covers 99.84% of the training tokens. word2vec, used in the course exercise, covers only about 78%: it has no vectors for punctuation and frequent words like *to*, *a*, *and*, *of*. fastText covers 99.38%, but has no vector for `n't`, so all negations (*didn't*, *wasn't*) would be lost. So GloVe is more suitable for this project.


## Stopwords 
Download NLTK's English stopword list and check which important words it contains.

In [9]:
nltk.download('stopwords')
stopwords = set(nltk.corpus.stopwords.words('english'))
print('Stopwords:', len(stopwords))
print([word for word in ['not', 'no', 'never', 'before', 'after', 'then'] if word in stopwords])

Stopwords: 198
['not', 'no', 'before', 'after', 'then']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mmici\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


How many words in the texts are stopwords? Share of the training tokens that are stopwords.

In [10]:
stopword_tokens = sum(fdist_lower[word] for word in fdist_lower if word in stopwords)
print(f'Stopword tokens: {stopword_tokens / fdist_lower.N():.2%}')

Stopword tokens: 38.95%


Number of hypothesis pairs in which h1 and h2 differ only in stopwords. 

In [11]:
only_stopwords = 0
for h1, h2 in zip(train['hypothesis_1'], train['hypothesis_2']):
    words_1 = set(word.lower() for word in nltk.word_tokenize(h1))
    words_2 = set(word.lower() for word in nltk.word_tokenize(h2))
    difference = words_1 ^ words_2  # words that occur in only one of the two hypotheses
    if difference and difference <= stopwords:
        only_stopwords += 1 
print('Pairs that differ only in stopwords:', only_stopwords)
print(f'Share: {only_stopwords / len(train):.2%}')

Pairs that differ only in stopwords: 367
Share: 2.18%


NLTK's stopword list contains *not*, *no*, *before*, *after* and *then*, words that change the meaning of a sentence. Stopwords make up 38.95% of the training tokens, and in 367 pairs (2.18%) the two hypotheses differ only in stopwords; after removing them, h1 and h2 would be identical. **Stopwords are therefore kept.**

## Preprocessing function
All decisions above in one function: tokenize with NLTK, lowercase all words, keep stopwords.

In [12]:
def preprocess(text):
    # tokenize
    text = nltk.word_tokenize(text)
    # lowercase all words
    text = [word.lower() for word in text]
    # remove stopwords (not used, see above)
    # text = [word for word in text if word not in stopwords]
    return text

print(train[269]["hypothesis_1"])
print(preprocess(train[269]["hypothesis_1"]))

He didn't listen to them, and had to go to the hospital.
['he', 'did', "n't", 'listen', 'to', 'them', ',', 'and', 'had', 'to', 'go', 'to', 'the', 'hospital', '.']


## Sentence vectors
Each text becomes one vector: the average of the GloVe vectors of its words (course slides "Word Embeddings", slide 32). Words without a GloVe vector are skipped.

In [13]:
words = preprocess(train[269]['hypothesis_1'])
vectors = [wv[word] for word in words if word in wv]
print(len(words), len(vectors))
print(vectors[0].shape)

15 15
(300,)


Average the word vectors: one vector with 300 numbers for the whole sentence.

In [14]:
import numpy as np

sentence_vector = np.mean(vectors, axis=0)
print(sentence_vector.shape)
print(sentence_vector[:5])

(300,)
[-0.21954928  0.03110227 -0.24757503 -0.11346499 -0.11204454]


The same steps as a function that works for any text. A text without any known word in GloVe gets a zero vector.

In [15]:
def average_vector(text):
    words = preprocess(text)
    vectors = [wv[word] for word in words if word in wv]
    # no known word: zero vector
    if len(vectors) == 0:
        return np.zeros(wv.vector_size)
    return np.mean(vectors, axis=0)

print(average_vector(train[269]["hypothesis_1"])[:5])
print(average_vector("mturk").shape)
print(average_vector("mturk")[:5])

[-0.21954928  0.03110227 -0.24757503 -0.11346499 -0.11204454]
(300,)
[0. 0. 0. 0. 0.]


## Input format
One example becomes one input: the sentence vectors of O1, O2, h1 and h2 side by side (4 x 300 = 1200 numbers). The label (1 or 2) becomes the class 0 or 1.

In [16]:
columns = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

example = train[0]
x = np.concatenate([average_vector(example[column]) for column in columns])
y = example["label"] - 1
print(x.shape)
print(example["label"], y)

(1200,)
2 1


Apply this to all examples of each split: one row of 1200 numbers per example, and one class per example.

In [17]:
def example_vector(example):
    return np.concatenate([average_vector(example[column]) for column in columns])

X_train = np.array([example_vector(example) for example in train])
y_train = np.array([example["label"] - 1 for example in train])
print(X_train.shape, y_train.shape)

(16801, 1200) (16801,)


In [18]:
X_valid = np.array([example_vector(example) for example in valid])
y_valid = np.array([example["label"] - 1 for example in valid])
X_test = np.array([example_vector(example) for example in test])
y_test = np.array([example["label"] - 1 for example in test])
print(X_valid.shape, y_valid.shape)
print(X_test.shape, y_test.shape)

(1000, 1200) (1000,)
(1532, 1200) (1532,)


Save the features and labels of all splits to one file, so that the training notebooks can load them without GloVe.

In [19]:
import os

os.makedirs("features", exist_ok=True)
np.savez("features/model1.npz",
         X_train=X_train.astype(np.float32), y_train=y_train,
         X_valid=X_valid.astype(np.float32), y_valid=y_valid,
         X_test=X_test.astype(np.float32), y_test=y_test)
print(os.path.getsize("features/model1.npz") / 1e6, "MB")

92.954558 MB


## Model 2: text lengths in tokens
Model 2 reads each text word by word, and all texts in a batch must have the same length (padding). Measure the length of the training texts in tokens after preprocessing.

In [20]:
# Model 2 reads every text word by word, so the length in tokens matters
for column in columns:
    # number of tokens of each training text in this column (same preprocess() as model 1)
    lengths = [len(preprocess(text)) for text in train[column]]
    # average length, longest text, and the length that 99% of the texts do not exceed
    print(f'{column}: mean {np.mean(lengths):.1f}, max {max(lengths)}, 99% <= {np.percentile(lengths, 99):.0f}')

observation_1: mean 9.2, max 20, 99% <= 15
observation_2: mean 10.6, max 20, 99% <= 17
hypothesis_1: mean 9.6, max 34, 99% <= 20
hypothesis_2: mean 9.6, max 33, 99% <= 21


## Model 2: padding
The texts in a batch must have the same length. All texts are padded to the length of the longest training text, 34 tokens, so no training word is lost; longer texts in the validation and test sets are cut. The padding (number 0) is added at the end of each text, as in the course slides "Project Discussions" (Week 4, slide 13). In model 2, `pack_padded_sequence` (same slides, slide 17) passes the real length of each text to the LSTM, so the LSTM stops after the last real word and ignores the padding; its final hidden state h_n is then the state after the last word.


In [21]:
# all texts get the length of the longest training text (see the output above)
max_length = 34

def pad(numbers):
    # cut texts that are longer than max_length (only happens in validation and test)
    numbers = numbers[:max_length]
    # zeros (padding) at the end, so that the real words come first
    return numbers + [0] * (max_length - len(numbers))

# test with made-up word numbers (the real ones come from the vocabulary, next step)
padded = pad([14, 102, 57, 1893, 3])
print(padded)
print(len(padded))


[14, 102, 57, 1893, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
34


## Model 2: vocabulary
Each word gets a number, so that a text becomes a list of numbers. The vocabulary is built from the training data only: every lowercased training word that has a GloVe vector, sorted by frequency (the most frequent word gets number 1). Number 0 is kept for the padding. Rare words are kept: their vectors come from GloVe, which was trained on a much larger corpus, so a word that is rare in this dataset still has a useful vector. Words without a GloVe vector are skipped, as in model 1.

In [22]:
# vocabulary: word -> number, for every training word that has a GloVe vector
vocab = {}
# most_common(): all training words, the most frequent first
for word, count in fdist_lower.most_common():
    # words without a GloVe vector are skipped (as in model 1)
    if word in wv:
        # next free number: 1 for the first word, 2 for the second, ... (0 stays free for padding)
        vocab[word] = len(vocab) + 1

print('Vocabulary size:', len(vocab))
# the first 5 entries (word, number)
print(list(vocab.items())[:5])
# look up the number of one word
print(vocab['hospital'])

Vocabulary size: 15464
[('.', 1), ('the', 2), ('to', 3), ('a', 4), ('was', 5)]
527


## Model 2: text to numbers
Each text is preprocessed as for model 1, every word is replaced by its number in the vocabulary (words that are not in the vocabulary are skipped), and the list is padded to 34 numbers.


In [23]:
def to_numbers(text):
    # same preprocessing as model 1: tokenize, lowercase
    words = preprocess(text)
    # the number of each word; words that are not in the vocabulary are skipped
    numbers = [vocab[word] for word in words if word in vocab]
    # zeros at the end, up to 34 numbers
    return pad(numbers)

# test with the example sentence from the tokenization section
text = train[269]['hypothesis_1']
print(text)
print(to_numbers(text))


He didn't listen to them, and had to go to the hospital.
[8, 28, 23, 1296, 3, 73, 13, 6, 17, 3, 47, 3, 2, 527, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## Model 2: embedding matrix
A table with one row per number: row i is the GloVe vector of the word with number i, and row 0 (padding) stays all zeros. In model 2, the embedding layer looks up the word vectors in this table. It is frozen, so the GloVe vectors are not changed during training (as in model 1).


In [24]:
# one row per number (0 = padding, 1 ... 15464 = words), 300 columns (the GloVe vector size)
embedding_matrix = np.zeros((len(vocab) + 1, wv.vector_size), dtype=np.float32)
# fill row "number" with the GloVe vector of the word; row 0 stays all zeros
for word, number in vocab.items():
    embedding_matrix[number] = wv[word]

print(embedding_matrix.shape)
# row 0 (padding): first 5 numbers
print(embedding_matrix[0][:5])
# row of "hospital" and the GloVe vector of "hospital": should be the same
print(embedding_matrix[vocab['hospital']][:5])
print(wv['hospital'][:5])


(15465, 300)
[0. 0. 0. 0. 0.]
[-0.87315   -0.62477   -0.096836   0.0025302 -0.010344 ]
[-0.87315   -0.62477   -0.096836   0.0025302 -0.010344 ]


## Model 2: input format
One example becomes 4 rows of 34 word numbers: O1, O2, h1 and h2, in the same order as for model 1. Apply this to all examples of each split (one array of shape [examples, 4, 34] per split). The numbers are stored as whole numbers (`int64`), because the embedding layer needs indices. The labels are the same as for model 1.

In [25]:
# one example: 4 rows of 34 word numbers (O1, O2, h1, h2, in the order of `columns`)
def example_numbers(example):
    return [to_numbers(example[column]) for column in columns]

# all examples of each split, shape (examples, 4, 34)
# X2 = input of model 2 (X_train, X_valid, X_test above are the input of model 1)
X2_train = np.array([example_numbers(example) for example in train], dtype=np.int64)
X2_valid = np.array([example_numbers(example) for example in valid], dtype=np.int64)
X2_test = np.array([example_numbers(example) for example in test], dtype=np.int64)
print(X2_train.shape, X2_valid.shape, X2_test.shape)
# check: h1 of train[269] (row 2) is the same as in the test of to_numbers() above
print(X2_train[269][2])

(16801, 4, 34) (1000, 4, 34) (1532, 4, 34)
[   8   28   23 1296    3   73   13    6   17    3   47    3    2  527
    1    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]


Length of each text in word numbers (all numbers that are not 0). `pack_padded_sequence` in model 2 needs a length of at least 1 for every text, so check the shortest text of each split.

In [26]:
for name, X2 in [('train', X2_train), ('valid', X2_valid), ('test', X2_test)]:
    # number of real word numbers (not 0 = padding) in every text, shape (examples, 4)
    lengths = (X2 != 0).sum(axis=-1)
    print(f'{name}: shortest text {lengths.min()}, longest text {lengths.max()} word numbers')

train: shortest text 1, longest text 34 word numbers
valid: shortest text 2, longest text 34 word numbers
test: shortest text 2, longest text 28 word numbers


Share of the tokens of each split that are in the vocabulary. Words that are not in it are skipped: in the training data only words without a GloVe vector, in the validation and test data also words that never occur in the training data.

In [27]:
for name, split in [('train', train), ('valid', valid), ('test', test)]:
    # all tokens of the split (O1, O2, h1 and h2 of every example)
    words = [word for column in columns for text in split[column] for word in preprocess(text)]
    # count the tokens that have a number in the vocabulary (True counts as 1)
    known = sum(word in vocab for word in words)
    print(f'{name}: {known / len(words):.2%} of the tokens are in the vocabulary')

train: 99.84% of the tokens are in the vocabulary
valid: 98.45% of the tokens are in the vocabulary
test: 98.43% of the tokens are in the vocabulary


Almost all tokens are kept: 99.84% of the training tokens and 98.45% / 98.43% of the validation / test tokens. About 1.5% of the validation and test tokens are skipped because the words never occur in the training data; model 1 still used them if they had a GloVe vector. This is the cost of building the vocabulary from the training data only.

Save the word numbers, the labels and the embedding matrix of all splits to one file, so that the training notebook of model 2 can load them without GloVe (as for model 1).

In [28]:
# same format as for model 1; the embedding matrix is saved too, for the embedding layer of model 2
np.savez("features/model2.npz",
         X_train=X2_train, y_train=y_train,
         X_valid=X2_valid, y_valid=y_valid,
         X_test=X2_test, y_test=y_test,
         embedding_matrix=embedding_matrix)
print(os.path.getsize("features/model2.npz") / 1e6, "MB")

39.748726 MB


## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Dataset: course version of ART, `amarfurt/art-nlp-hs26` on Hugging Face (A. Marfurt), based on Bhagavatula et al. (2020), *Abductive Commonsense Reasoning*, ICLR 2020, https://arxiv.org/abs/1908.05739.
- Loading the splits: code from slide 9 of the course slides "Course Project HS26" (A. Marfurt).
- Tokenization: `nltk.download('punkt_tab')` and `nltk.word_tokenize` as in the course exercise "NLP Pipeline" (Week 1).
- Tokens/types and `nltk.FreqDist` as in the course exercise "NLP Pipeline" (Week 1); coverage check (`word in wv`).
- Word embeddings: GloVe (Pennington et al., 2014), fastText (Bojanowski et al., 2017) and word2vec (Mikolov et al., 2013), loaded with gensim's downloader (Řehůřek & Sojka, 2010) as in the course exercise "Word Embeddings" (Week 2).
- Stopwords: NLTK's English stopword list (`nltk.corpus.stopwords`) as in the course exercise "Topic Modeling" (Week 2).
- Preprocessing function: structure of `preprocess()` as in the course exercise "Topic Modeling" (Week 2).
- Sentence vectors: averaging the word vectors of a text as input for a classifier, from the course slides "Word Embeddings" (Week 2), slide 32 "Extrinsic Evaluation"; word vector lookup (`wv[word]`) as in the course exercise "Word Embeddings" (Week 2); `np.mean`, `np.zeros`, `np.concatenate` and `np.array` from NumPy (Harris et al., 2020), https://numpy.org/doc/stable/.
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Saving the features: `np.savez` and `ndarray.astype` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html and https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html; `os.makedirs` and `os.path.getsize` from the Python standard library, https://docs.python.org/3/library/os.html.
- Text lengths: `np.percentile` from NumPy (Harris et al., 2020), https://numpy.org/doc/stable/reference/generated/numpy.percentile.html.
- Padding: a batch has one sequence length, so texts of different lengths are filled up (course slides "Preliminaries" (Week 1), slides 14-17); padding at the end of the text, pad tokens ignored in the computation, and `pack_padded_sequence` for padding in RNNs from the course slides "Project Discussions" (Week 4, "Data Loading"), slides 13 and 17, and the PyTorch documentation `torch.nn.utils.rnn.pack_padded_sequence`, https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html; list slicing, `*` and `+` from the Python documentation "Common Sequence Operations", https://docs.python.org/3/library/stdtypes.html#common-sequence-operations; padding and truncation to a fixed length as with `padding="max_length", truncation=True` of the tokenizer in the course exercise "Experiment Tracking".
- Vocabulary: each word gets its position in the vocabulary, as for one-hot vectors in the course slides "Vector Space Model" (Week 2), slide 5; built from the training words with `nltk.FreqDist` and `most_common()` as in the course exercise "NLP Pipeline" (Week 1); `word in wv` as in the course exercise "Word Embeddings" (Week 2); Python dictionaries, https://docs.python.org/3/tutorial/datastructures.html#dictionaries.
- Text to numbers: list comprehension with an `if` condition, Python tutorial "List Comprehensions", https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions.
- Embedding matrix: input embedding matrix of size vocabulary size x vector size from the course slides "Preliminaries" (Week 1), slide 18; looking up a row is the same as multiplying a one-hot vector with the matrix, as in the word2vec architecture (course slides "Word Embeddings" (Week 2), slide 8); pretrained word vectors as the embedding matrix of a neural network, course slides "Word Embeddings" (Week 2), slide 23; `np.zeros` with `shape` and `dtype` from the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.zeros.html; `dict.items()` from the Python tutorial "Looping Techniques", https://docs.python.org/3/tutorial/datastructures.html#looping-techniques; word vector lookup (`wv[word]`) as in the course exercise "Word Embeddings" (Week 2); row 0 for padding as `padding_idx` in the PyTorch documentation `torch.nn.Embedding`, https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html.
- Input of model 2: arrays of word numbers with `np.array(..., dtype=np.int64)`, because the embedding layer needs indices (PyTorch documentation `torch.nn.Embedding`: input "IntTensor or LongTensor ... containing the indices to extract"); nested list comprehension from the Python tutorial "List Comprehensions"; counting with `sum()` over True/False values; saving with `np.savez` as for model 1 (see above). Text lengths for `pack_padded_sequence`: counting the non-zero numbers with `(X2 != 0).sum(axis=-1)` (NumPy documentation `numpy.sum`, https://numpy.org/doc/stable/reference/generated/numpy.sum.html).
- Tools: code written with the help of Claude (Anthropic); all code reviewed and understood by me.
